In [ ]:
import torch

from datasets import load_dataset
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

from pathlib import Path

# Use the install_model.py if the notebook does not work. It tends to break the download.
## The errors are fine, just the reality of modern machine learning.

In [ ]:
# 1. Qwen2.5 Coder 7B Instruct
# MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

# 2. Qwen3.5 4B
MODEL_NAME = "Qwen/Qwen3.5-4B"

# 3. SFT model
# MODEL_NAME = "Benyucong/sft_quantum_circuit_gen_4B"

# 4. Qwen3.5 9B
# MODEL_NAME = "Qwen/Qwen3.5-9B"

In [ ]:
DATASET_PATH = Path("quantum_optimization_dataset.jsonl")
ADAPTER_DIR = Path("qwen35_4b_quantum_optimizer_qlora")

TEST_SIZE = 0.05
SEED = 42

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATASET_PATH.resolve()}\n"
        "Set DATASET_PATH to the JSONL file created by data_set_creator.ipynb."
    )

In [ ]:
SYSTEM_PROMPT = (
    "You are a quantum circuit compiler. "
    "Rewrite the supplied OpenQASM 3 program into a mathematically "
    "equivalent but simpler OpenQASM 3 program. "
    "The output must implement exactly the same unitary transformation. "
    "Every qubit used in the output must be declared. "
    "Do not introduce measurements, classical bits, resets, or classical "
    "control when the input contains none. "
    "Do not remove qubit declarations. "
    "Return a complete, syntactically valid OpenQASM 3 program and nothing else."
)

USER_PREFIX = (
    "Optimize this quantum circuit by reducing gate count and depth "
    "where possible while preserving its exact operation:\n\n"
)

raw_dataset = load_dataset(
    "json",
    data_files=str(DATASET_PATH),
    split="train",
)

required_columns = {"input", "output"}
missing = required_columns - set(raw_dataset.column_names)
if missing:
    raise ValueError(
        f"Dataset is missing required columns: {sorted(missing)}. "
        f"Found: {raw_dataset.column_names}"
    )

def to_prompt_completion(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PREFIX + example["input"]},
        ],
        "completion": [
            {"role": "assistant", "content": example["output"]},
        ],
    }

formatted_dataset = raw_dataset.map(
    to_prompt_completion,
    remove_columns=raw_dataset.column_names,
)

split = formatted_dataset.train_test_split(
    test_size=TEST_SIZE,
    seed=SEED,
    shuffle=True,
)

train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Training examples:   {len(train_dataset):,}")
print(f"Validation examples: {len(eval_dataset):,}")

### Load Qwen3.5-9B in 4-bit and attach LoRA

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "QLoRA of a 9B model is intended to run on a CUDA GPU. "
        "No CUDA device was detected."
    )

USE_BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print("GPU:", torch.cuda.get_device_name(0))
print("QLoRA compute dtype:", COMPUTE_DTYPE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

try:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map={"": 0},
        torch_dtype=COMPUTE_DTYPE,
    )
except (ImportError, ValueError, TypeError):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map={"": 0},
        torch_dtype=COMPUTE_DTYPE,
    )

model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
    # Qwen3.5-9B also contains a vision tower. We only want text-side adapters.
    exclude_modules=r".*(visual|vision).*",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = SFTConfig(
    output_dir=str(ADAPTER_DIR),

    num_train_epochs=1,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=1e-4,
    warmup_steps=5,

    logging_steps=5,

    save_strategy="no",

    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),

    gradient_checkpointing=True,

    max_length=512,

    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

In [ ]:
train_result = trainer.train()
train_result

In [ ]:
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

print(f"Saved QLoRA adapter to: {ADAPTER_DIR.resolve()}")